# DD-PRiSM-plus — Step 3: train the paper's model

**Drug representation: `morgan`**

The published representation: a **512-bit Morgan fingerprint**, radius 2. This is the baseline every other experiment is measured against, and it is the run that reproduces the paper.

**GPU session.** Accelerator → **GPU T4 x2**, Internet → **On**.

**Attach step 2's output:** right panel → **Add Input → Your Work →**
**Notebook Output**, pick your `02_preprocess` version.

### What to look for

Reproduces the published numbers. On the unseen pair set we measured RMSE 0.0828 / PCC 0.9386 against the paper's 0.0830 / 0.9387.

---

Three stages run in order, all on the authors' own model classes from
`original/ddprism_original.py`:

| stage | data | learning rate |
|---|---|---|
| pretrain | NCI60, ~8.9M rows | 1e-2 |
| finetune | ALMANAC monotherapy, 35k rows, curve prediction network unfrozen | 1e-3 |
| combination | ALMANAC pairs, 1.98M rows | 1e-2 |

> **This may not finish in one session.** Kaggle stops you at ~12 hours.
> Every epoch is checkpointed, so just Save Version, attach *this* notebook's
> output next time, and rerun — it resumes at the epoch it reached.

In [ ]:
import os, glob

REPO = '/kaggle/working/ddprism-plus'

FEATURES = 'morgan'
RUNS     = '/kaggle/working/runs'

# This notebook owns this directory and no other. Two representations
# produce models of different shapes, so sharing one would try to resume a
# 512-wide model from an 896-wide checkpoint -- or silently overwrite the
# baseline this experiment is meant to be measured against.
print('features:', FEATURES)
print('runs    :', RUNS)

if os.path.exists(REPO):
    !cd {REPO} && git pull --quiet
else:
    !git clone --quiet https://github.com/SanaNiroomand/DD-PRiSM-plus.git {REPO}
os.chdir(REPO)

hits = glob.glob('/kaggle/input/**/nci60_filtered.parquet', recursive=True)
if not hits:
    raise SystemExit('preprocessed data not found -- attach the output of '
                     '02_preprocess via Add Input.')
PROCESSED = os.path.dirname(hits[0])

raw = glob.glob('/kaggle/input/**/DOSERESP.zip', recursive=True)
if not raw:
    raise SystemExit('raw data not found -- attach 01_setup_and_data too, '
                     'for the KEGG .gmt.')
DATA = os.path.dirname(raw[0])

print('processed:', PROCESSED)
print('raw      :', DATA)

## Resume from a previous session

**Save & Run All always starts a fresh container**, so a previous run's
results only survive if you feed them back in: **Add Input → Your Work →**
**Notebook Output**, and pick this notebook's own earlier version. This cell
then copies its checkpoints into place. Does nothing on a first run.

`REDO` deletes a stage's checkpoint so it trains from scratch, while keeping
the stages before it. Without that, a restored checkpoint is resumed -- a
stage that already early-stopped would run one epoch, stop again, and change
nothing. Its `best.pt` is kept, since the next stage loads weights from it
until the retrain overwrites them.

In [ ]:
import shutil

REDO = []          # e.g. ['finetune', 'combination'] to retrain those two

os.makedirs(RUNS, exist_ok=True)
restored = 0
# Only this experiment's own checkpoints, from runs/ and nowhere else.
pattern = '/kaggle/input/**/runs/*/checkpoint.pt'
for found in glob.glob(pattern, recursive=True):
    stage = os.path.basename(os.path.dirname(found))
    os.makedirs(os.path.join(RUNS, stage), exist_ok=True)
    for name in ('checkpoint.pt', 'best.pt', 'history.json'):
        source = os.path.join(os.path.dirname(found), name)
        if os.path.exists(source):
            shutil.copy(source, os.path.join(RUNS, stage, name))
            restored += 1
for config in glob.glob('/kaggle/input/**/runs/config.json', recursive=True):
    shutil.copy(config, os.path.join(RUNS, 'config.json'))
print('restored', restored, 'checkpoint files')

for stage in REDO:
    stale = os.path.join(RUNS, stage, 'checkpoint.pt')
    if os.path.exists(stale):
        os.remove(stale)
        print('cleared', stage, '-- it will train from scratch')

!ls -R {RUNS} 2>/dev/null | head -20

## Install and check

In [ ]:
!pip install --quiet zipfile-deflate64 pyarrow
!python -m pytest tests -q -x --ignore=tests/test_train.py

## Train

`original` is the authors' loop over 186 pathways; `fast` batches those 186
steps and is pinned to the published model at 1e-10 by the test suite, and
measured at 1.3e-15 on a T4. They compute the same thing.

The difference is only speed, but at this scale speed decides whether you
finish. Measured on a T4, per NCI60 epoch:

| model | min/epoch | epochs in 30 GPU-hours |
|---|---|---|
| `original` | 31.5 | **57** |
| `fast` | 6.0 | **300** |

`--max-hours` stops cleanly and checkpoints before Kaggle pulls the plug.

In [ ]:
MODEL = 'fast'      # 'original' for the authors' loop, 'fast' for the same maths batched
STAGE = 'all'       # or e.g. 'finetune combination' to redo just those two

INIT_FROM = None

ARGS = (f'--data {DATA} --processed {PROCESSED} --out {RUNS} '
        f'--model {MODEL} --stage {STAGE} --drug-features {FEATURES}')
if INIT_FROM:
    ARGS += f' --init-from {INIT_FROM}'

!python -m ddprism.train {ARGS} --batch-size 1024 --max-hours 10.5

## Training curve

In [ ]:
import json
for stage in ('pretrain', 'finetune', 'combination'):
    path = os.path.join(RUNS, stage, 'history.json')
    if not os.path.exists(path):
        continue
    history = json.load(open(path))
    best = min(history, key=lambda h: h['val_loss'])
    print(f"{stage:<12} {len(history):>3} epochs   best val {best['val_loss']:.5f}  "
          f"RMSE {best['val_rmse']:.4f}  PCC {best['val_pcc']:.4f}")
print()
print('paper, unseen pair set:')
print('  pretrained  RMSE 0.0830  PCC 0.9387')
print('  fine-tuned  RMSE 0.0914  PCC 0.8791')
print('  combination RMSE 0.0854  PCC 0.9063')

## Score on the held-out sets — the numbers to quote

Training reports on a random slice of its own pool, which splits individual
measurements: the same drug on the same cell line can be in training at one
dose and validation at another. The paper's figures hold out whole entities.
This scores the trained model on those, so the comparison is like for like.

No training, one forward pass per split.

In [ ]:
!python -m ddprism.evaluate --data {DATA} --processed {PROCESSED} --runs {RUNS} --drug-features {FEATURES}

## Save

**Save Version → Save & Run All (Commit).**

This notebook's output holds `evaluation.json` for this experiment alone.
`06_compare` reads it, together with the other experiments', and puts them
side by side.

If training stopped on the time budget rather than finishing, attach this
notebook's own output next session and rerun. It picks up mid-stage.